In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pylab as plt

%matplotlib inline 
plt.style.use('seaborn-whitegrid')
plt.rc('text', usetex=True)
plt.rc('font', family='times')
plt.rc('xtick', labelsize=10) 
plt.rc('ytick', labelsize=10) 
plt.rc('font', size=12) 
plt.rc('figure', figsize = (12, 5))

In [2]:
import numpy as np
import pandas as pd
from sklearn import cluster
from sklearn import metrics
from scipy.spatial.distance import cdist
from yellowbrick.cluster import KElbowVisualizer

In [3]:
df = pd.read_csv('100_submission_measures_binary.csv')
df

,order,id,user,width,date_submission,first_reply,depth,last_reply,score,size,...,cause,discrep,certain,differ,money,article,conj,filler,adverb,function
0,1,abeb43,mraza007,3,2019-01-01 03:01:01,2019-01-01 03:11:02,2,2019-01-01 03:50:43,1,4,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0
1,2,abed2v,CafeRoaster,1,2019-01-01 03:08:27,2019-01-01 21:45:05,4,2019-01-03 02:13:39,2,4,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
2,4,abemi9,introverted_rabbit,109,2019-01-01 03:45:21,2019-01-01 03:53:20,13,2019-01-23 23:27:51,489,210,...,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0
3,5,abeyaw,yiaux,4,2019-01-01 04:32:32,2019-01-01 07:07:16,5,2019-01-14 19:20:17,2,10,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,7,abfbnd,SkepticDad17,3,2019-01-01 05:19:17,2019-01-01 05:40:43,3,2019-01-01 09:19:39,0,4,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26283,30073,eice3p,Schopenhaur1859,2,2020-01-01 01:58:04,2020-01-01 02:12:33,1,2020-01-01 09:47:28,1,2,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
26284,30074,eicety,textssg,1,2020-01-01 02:00:00,2020-01-01 02:14:21,2,2020-01-02 18:42:41,1,2,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
26285,30075,eicjfd,oogaboogabebeamasid,3,2020-01-01 02:11:31,2020-01-01 02:18:55,3,2020-01-02 06:46:02,2,5,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
26286,30076,eicypb,Trifonas-Kaoulla,2,2020-01-01 02:51:46,2020-01-01 02:56:02,6,2020-01-01 03:28:48,1,8,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0


In [4]:
feat_social = ['size', 'i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 'negemo', 'work', 'power', 'drives', 
               'percept', 'negate', 'interrog', 'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 
               'affiliation', 'social']

df_social = df[feat_social]
df_social.drop(['size'], axis=1)

,i,we,pronoun,ppron,ipron,affect,posemo,negemo,work,power,...,percept,negate,interrog,focuspresent,auxverb,you,assent,focuspast,affiliation,social
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
4,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26283,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
26284,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
26285,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
26286,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0


In [ ]:
df_metrics = pd.DataFrame(columns = ['K', 'size', 'num_subs', 'distortion_min', 'distortion_avg', 'inertia', 'silhouette'])

In [ ]:
for K in range (2, 6):
    df_social = df[feat_social]
    for sub_size in range(0, 1):
        df_social.drop(df_social[df_social['size'] <= sub_size].index, inplace = True)
        print(str(K) + ', ' + str(sub_size))
        clf = cluster.KMeans(n_clusters=K, init='k-means++', random_state=0, max_iter=300, n_init=10)
        clf.fit(df_social.drop(['size'], axis=1))
        df_metrics = df_metrics.append({'K': K, 
                                        'size': sub_size,
                                        'num_subs': df_social.shape[0], 
                                        'distortion_min': sum(np.min(cdist(df_social.drop(['size'], axis=1), clf.cluster_centers_, 'euclidean'),axis=1)) / df_social.drop(['size'], axis=1).shape[0],
                                        'distortion_avg': sum(np.average(cdist(df_social.drop(['size'], axis=1), clf.cluster_centers_, 'euclidean'),axis=1)) / df_social.drop(['size'], axis=1).shape[0],
                                        'inertia': clf.inertia_, 
                                        'silhouette': metrics.silhouette_score(df_social.drop(['size'], axis=1), clf.labels_, metric='euclidean') }, 
                                       ignore_index=True)


In [ ]:
df_metrics.to_csv(path_or_buf='kmeans_metrics_K1.csv')

In [ ]:
df_metrics

In [ ]:
#df_metrics = pd.read_csv('kmeans_metrics_mean.csv')
#df_elbow = df_metrics.drop(df_metrics[df_metrics['size'] > 0.0].index)

fig = plt.figure(figsize=(15, 5))
plt.plot(range(2, 11), df_metrics['distortion_min'])
plt.grid(True)
plt.title('Elbow curve')

In [ ]:
model = cluster.KMeans()
visualizer = KElbowVisualizer(model, k=(2,10))
visualizer.fit(df_social.drop(['size'], axis=1))
visualizer.show(outpath="kelbow_minibatchkmeans.png")

In [ ]:
plt.savefig('fig.png')

In [ ]:
clf.labels_

# Reinício

In [5]:
feat_social = ['size', 'i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 'negemo', 'work', 'power', 'drives', 
               'percept', 'negate', 'interrog', 'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 
               'affiliation', 'social']

df = pd.read_csv('100_submission_measures_binary.csv')
#df.drop(df[df['size'] <= 1].index, inplace=True)

df_social = df[feat_social]
df_social.drop(df_social[df_social['size'] <= 3].index, inplace = True)
df_social.drop(['size'], axis=1, inplace=True)
df_social

,i,we,pronoun,ppron,ipron,affect,posemo,negemo,work,power,...,percept,negate,interrog,focuspresent,auxverb,you,assent,focuspast,affiliation,social
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
4,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26281,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0
26282,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
26285,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
26286,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0


In [ ]:
model = cluster.KMeans()
visualizer = KElbowVisualizer(model, k=(2,11))
visualizer.fit(df_social)
visualizer.show()

In [6]:
clf = cluster.KMeans(n_clusters=4, random_state=0, max_iter=300, n_init=10)
clf.fit(df_social)

KMeans(n_clusters=4, random_state=0)

In [8]:
pd.DataFrame(list(
            zip(
                ['inertia', 'silhouette'], 
                [
                    clf.inertia_, 
                    metrics.silhouette_score(df_social, clf.labels_, metric='euclidean')
                ]
                )
            ), 
            columns = ['measures', 'coeff']).to_csv('140_submission_coefficients.csv', index=False)

In [11]:
np.bincount(clf.labels_[clf.labels_>=0])

array([3098, 3304, 3319, 3792])

In [10]:
df.drop(df[df['size'] <= 3].index, inplace = True)
df['cluster'] = clf.labels_

In [14]:
df_cluster = df.loc[:, ['score_plus', 'size', 'number_participants', 'subcommunities.strong', 'bottlenecks', 'number_triads', 'cluster']]
df_cluster
test = df_cluster.groupby(['cluster']).agg({'score_plus': 'mean', 
                                            'size': 'mean', 
                                            'number_participants': 'mean', 
                                            'subcommunities.strong': 'mean', 
                                            'bottlenecks': 'mean', 
                                            'number_triads': 'mean'}).reset_index()

In [18]:
#test.insert(loc=1, column='cluster_size', value=np.bincount(clf.labels_[clf.labels_>=0]))
test

,cluster,cluster_size,score_plus,size,number_participants,subcommunities.strong,bottlenecks,number_triads
0,0,3098,128.806327,12.689477,8.122660,5.123305,1.865397,74.365720
1,1,3304,42.657990,8.668886,6.403753,4.918281,2.169794,35.461562
2,2,3319,56.084061,10.787888,6.449834,4.272371,1.899669,38.590238
3,3,3792,72.517141,11.664821,7.318565,5.097310,2.151899,40.434599


In [19]:
test.groupby(['cluster']).agg({'score_plus': 'mean'}).sort_values('score_plus', ascending=False).reset_index().loc[0, 'cluster']

0

In [ ]:
centers = np.append(clf.cluster_centers_, df_cluster.groupby(['cluster']).agg({'score_plus': 'mean'}), axis=1)

In [ ]:
centers[0,21]

In [ ]:
centers

In [ ]:
names = ['i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 'negemo', 'work', 'power', 'drives', 
         'percept', 'negate', 'interrog', 'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 
         'affiliation', 'social', 'score_plus']
res = pd.DataFrame(np.append(clf.cluster_centers_, df_cluster.groupby(['cluster']).agg({'score_plus': 'mean'}), 
                             axis=1), 
                  columns=names)

In [ ]:
res